In [4]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.tuning_ml_models import imcp_score_adapted
from src.cluster_undersampling import ClusterUnderSampler

import joblib
import numpy as np
import pandas as pd
from glob import glob
from imblearn.pipeline import Pipeline
# import stratified kfold cross validation
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    make_scorer
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.cluster import KMeans
from lightgbm import LGBMClassifier

In [3]:
PATH_TO_DATA = '../data/strains_embs/Ab17978_embeddings.npz'

split_indices = np.load("../data/strains_embs/train_test_splits_indices.npy", allow_pickle=True).item()

train_indices = split_indices["Ab17978"]["train"]
test_indices = split_indices["Ab17978"]["test"]

with np.load(PATH_TO_DATA) as data:
        data = data['arr_0']
        X_train = data[train_indices, :-1]
        y_train = data[train_indices, -1]

        X_test = data[test_indices, :-1]
        y_test = data[test_indices, -1]

In [7]:
lgbm_base = LGBMClassifier(random_state=42, n_jobs=1,
                            max_depth=5,
                              n_estimators=100,
                                learning_rate=0.1, verbose=0)
lgbm_balancing = LGBMClassifier(random_state=42, n_jobs=1, class_weight='balanced', verbose=0)

lgbm_undersampling = Pipeline([
    ('undersampler', ClusterUnderSampler(n_clusters=2, 
                                         reduction_percentage_class=0.1,
                                         clustering_model=KMeans(), 
                                         random_state=42)),
    ('classifier', LGBMClassifier(random_state=42, n_jobs=1)) 
])

lgbm_undersampling_balancing = Pipeline([
    ('undersampler', ClusterUnderSampler(n_clusters=2, 
                                         reduction_percentage_class=0.1,
                                         clustering_model=KMeans(), 
                                         random_state=42)),
    ('classifier', LGBMClassifier(random_state=42, class_weight='balanced', n_jobs=1)) 
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'roc_auc': make_scorer(roc_auc_score),
    'imcp': make_scorer(imcp_score_adapted),
    'mcc': make_scorer(matthews_corrcoef)
}



In [8]:
cv_undersampling = cross_validate(
    estimator=lgbm_undersampling,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

cv_base = cross_validate(
    estimator=lgbm_base,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)
cv_balancing = cross_validate(
    estimator=lgbm_balancing,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

cv_undersampling_balancing = cross_validate(
    estimator=lgbm_undersampling_balancing,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)


[LightGBM] [Info] Number of positive: 6675, number of negative: 9365
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.069013 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 107904
[LightGBM] [Info] Number of data points in the train set: 16040, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.416147 -> initscore=-0.338610
[LightGBM] [Info] Start training from score -0.338610


/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6674, number of negative: 9365
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068216 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 107904
[LightGBM] [Info] Number of data points in the train set: 16039, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.416111 -> initscore=-0.338760
[LightGBM] [Info] Start training from score -0.338760


/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6674, number of negative: 9365
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072302 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 107904
[LightGBM] [Info] Number of data points in the train set: 16039, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.416111 -> initscore=-0.338760
[LightGBM] [Info] Start training from score -0.338760


...

/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alexsy/miniconda3/envs/py_ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature nam

In [9]:
df_cv_base = pd.DataFrame(cv_base)
df_cv_balacing = pd.DataFrame(cv_balancing)
df_cv_undersampling = pd.DataFrame(cv_undersampling)
df_cv_undersampling_balancing = pd.DataFrame(cv_undersampling_balancing)

In [10]:
print("Metrics\tBase Model\tClass Balancing\tUndersampling\tUndersampling + Balancing")
for col in df_cv_base.columns:
    # If col correspond to acc,au-roc, precision, recall, imcp, multiply by 100
    if col in ['test_accuracy', 'test_precision', 'test_recall', 'test_roc_auc', 'test_imcp']:
        df_cv_base[col] *= 100
        df_cv_balacing[col] *= 100
        df_cv_undersampling[col] *= 100
        df_cv_undersampling_balancing[col] *= 100
    base_mean = df_cv_base[col].mean()
    base_std = df_cv_base[col].std()
    balancing_mean = df_cv_balacing[col].mean()
    balancing_std = df_cv_balacing[col].std()
    undersampling_mean = df_cv_undersampling[col].mean()
    undersampling_std = df_cv_undersampling[col].std()
    undersampling_balancing_mean = df_cv_undersampling_balancing[col].mean()
    undersampling_balancing_std = df_cv_undersampling_balancing[col].std()
    
    print(f'{col}\t{base_mean:.2f} ± {base_std:.2f}\t{balancing_mean:.2f} ± {balancing_std:.2f}\t{undersampling_mean:.2f} ± {undersampling_std:.2f}\t{undersampling_balancing_mean:.2f} ± {undersampling_balancing_std:.2f}')



Metrics	Base Model	Class Balancing	Undersampling	Undersampling + Balancing
fit_time	21.80 ± 0.78	26.59 ± 0.29	10.36 ± 0.23	8.45 ± 0.07
score_time	0.13 ± 0.00	0.15 ± 0.00	0.17 ± 0.02	0.15 ± 0.00
test_accuracy	93.41 ± 0.03	76.29 ± 0.26	78.36 ± 0.59	72.35 ± 0.62
test_precision	62.14 ± 8.02	17.94 ± 0.37	18.68 ± 0.58	16.20 ± 0.31
test_recall	2.25 ± 0.22	71.80 ± 1.43	67.16 ± 1.22	75.67 ± 1.18
test_roc_auc	51.08 ± 0.11	74.20 ± 0.77	73.16 ± 0.75	73.89 ± 0.54
test_imcp	51.08 ± 0.11	74.21 ± 0.77	73.16 ± 0.75	73.90 ± 0.54
test_mcc	0.11 ± 0.01	0.27 ± 0.01	0.27 ± 0.01	0.26 ± 0.01
